In [2]:
import pandas as pd
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Subset

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:

#Pytorch dataset
class FakeNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [4]:
def load_model():
  # setup device

  # Primarily using Colab's TPU, on occasion will use Colab's GPU runtimes for the neural networks and machine learning tasks but primarily using TPU for BERT
  try:
      import torch_xla.core.xla_model as xm
      device = xm.xla_device()
      print(f"✓ Using TPU: {device}")
      use_tpu = True
  except ImportError:
      use_tpu = False
      if torch.cuda.is_available():
          device = torch.device("cuda")
          print(f"✓ Using CUDA: {torch.cuda.get_device_name(0)}")
          print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
      else:
          device = torch.device("cpu")
          print("⚠ Using CPU (Training will be slow!)")

  # Load Pre-trained BERT Model
  print("\n" + "=" * 60)
  print("MODEL LOADING")
  print("=" * 60)

  model = BertForSequenceClassification.from_pretrained(
      'bert-base-uncased',
      num_labels=2
  )

  if not use_tpu:
      model.to(device)
      print(f"✓ BERT model loaded and moved to {device}")
  else:
      print("✓ BERT model loaded (TPU will handle device placement)")

  return model, device, use_tpu

In [5]:
def compute_metrics(pred):
    """Compute accuracy, precision, recall, F1"""
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }


In [6]:
def cross_validate_finetuning(full_train_dataset, test_dataset, compute_metrics_fn, use_tpu, device, n_splits=5):
    print("\n" + "=" * 60)
    print(f"STARTING {n_splits}-FOLD CROSS VALIDATION (Pre-tokenized Data)")
    print("=" * 60)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    # Get labels from the dataset object to perform stratified split
    labels = full_train_dataset.labels
    indices = np.arange(len(labels))

    fold_accuracies = []
    fold_f1_scores = []
    fold_test_results = []
    fold_models = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(indices, labels)):
        print(f"\n--- FOLD {fold + 1} ---")

        # Create subsets using indices
        train_subset = Subset(full_train_dataset, train_idx)
        val_subset = Subset(full_train_dataset, val_idx)

        # Load fresh model instance
        model, _, _ = load_model()

        training_args = TrainingArguments(
            output_dir=f'./results_fold_{fold+1}',
            num_train_epochs=3,
            per_device_train_batch_size=128,
            per_device_eval_batch_size=128,
            eval_strategy="steps",
            eval_steps=128,
            save_strategy="steps",
            save_steps=128,
            save_total_limit=2,
            report_to="none",
            optim="adamw_torch",
            logging_dir='./logs',
            logging_steps=128,
            metric_for_best_model="f1",
            load_best_model_at_end=True,
            weight_decay=0.01,
        )
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_subset,
            eval_dataset=val_subset,
            compute_metrics=compute_metrics_fn,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
        )

        trainer.train()

        # Validation evaluation
        eval_metrics = trainer.evaluate()
        fold_accuracies.append(eval_metrics['eval_accuracy'])
        fold_f1_scores.append(eval_metrics['eval_f1'])
        print(f"Fold {fold+1} Validation - Accuracy: {eval_metrics['eval_accuracy']:.4f}, F1: {eval_metrics['eval_f1']:.4f}")

        # Test set evaluation
        test_metrics = trainer.evaluate(eval_dataset=test_dataset)
        fold_test_results.append(test_metrics)
        print(f"Fold {fold+1} Test     - Accuracy: {test_metrics['eval_accuracy']:.4f}, F1: {test_metrics['eval_f1']:.4f}, Precision: {test_metrics['eval_precision']:.4f}, Recall: {test_metrics['eval_recall']:.4f}")

        # Store the model for later selection
        fold_models.append(model)

    # ---- Summary ----
    print("\n" + "=" * 60)
    print("CROSS-VALIDATION SUMMARY")
    print("=" * 60)
    print(f"{'Fold':<6} {'Val Acc':<10} {'Val F1':<10} {'Test Acc':<10} {'Test F1':<10} {'Test Prec':<10} {'Test Rec':<10}")
    print("-" * 66)
    for i in range(n_splits):
        t = fold_test_results[i]
        print(f"{i+1:<6} {fold_accuracies[i]:<10.4f} {fold_f1_scores[i]:<10.4f} {t['eval_accuracy']:<10.4f} {t['eval_f1']:<10.4f} {t['eval_precision']:<10.4f} {t['eval_recall']:<10.4f}")

    avg_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    avg_f1 = np.mean(fold_f1_scores)
    std_f1 = np.std(fold_f1_scores)

    print(f"\nAverage Validation Accuracy: {avg_acc:.4f} (Std Dev: {std_acc:.4f})")
    print(f"Average Validation F1-score: {avg_f1:.4f} (Std Dev: {std_f1:.4f})")

    # Select best fold by validation F1-score
    best_fold_idx = int(np.argmax(fold_f1_scores))
    best_test = fold_test_results[best_fold_idx]
    print(f"\n★ Best Fold: {best_fold_idx + 1} (Val F1: {fold_f1_scores[best_fold_idx]:.4f})")
    print(f"  Test Results — Accuracy: {best_test['eval_accuracy']:.4f}, F1: {best_test['eval_f1']:.4f}, Precision: {best_test['eval_precision']:.4f}, Recall: {best_test['eval_recall']:.4f}")

    return fold_models[best_fold_idx]

In [7]:
def save_model(model, tokenizer, output_path, use_tpu, device):
  print("\n" + "=" * 60)
  print("MODEL SAVING")
  print("=" * 60)

  output_dir = output_path

  # Move model to CPU before saving if it's on TPU
  if use_tpu:
      model.to("cpu")
      print("Moved model to CPU for saving.")

  model.save_pretrained(output_dir)
  tokenizer.save_pretrained(output_dir)
  print(f"✓ Model saved to {output_dir}")

  # Move model back to original device after saving
  if use_tpu:
      model.to(device)
      print("Moved model back to TPU.")

  print("\n" + "=" * 60)
  print("BERT FINE-TUNING COMPLETE! 🎉")
  print("=" * 60)

In [8]:
def cross_dataset_evaluation(model, tokenizer, current_dataset_name, all_datasets_paths, compute_metrics_fn):
    print("\n" + "!" * 60)
    print(f"CROSS-DATASET GENERALIZATION: {current_dataset_name}")
    print("!" * 60)

    results = {}

    for name, path in all_datasets_paths.items():
        if name == current_dataset_name:
            continue

        print(f"\nTesting on unseen dataset: {name} (Full Dataset)...")
        # Load full unseen dataset for testing
        df = pd.read_csv(path).dropna()

        test_texts = df['combined_text'].tolist()
        test_labels = df['label'].tolist()

        encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)
        dataset = FakeNewsDataset(encodings, test_labels)

        # Trainer for evaluation
        eval_trainer = Trainer(
            model=model,
            compute_metrics=compute_metrics_fn
        )

        metrics = eval_trainer.evaluate(eval_dataset=dataset)
        results[name] = metrics
        print(f"  -> {name} Accuracy: {metrics['eval_accuracy']:.4f}, F1: {metrics['eval_f1']:.4f}")

    return results

In [9]:
# Dictionary of all 5 datasets
DATASETS = {
    "WELFake": "/content/drive/MyDrive/datasets/WELFake_processed.csv",
    "FakeNewsNet": "/content/drive/MyDrive/datasets/FakeNewsNet_processed.csv",
    "Fake_News_Detection": "/content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv",
    "ISOT": "/content/drive/MyDrive/datasets/ISOT_processed.csv",
    "Fake_News_Classification": "/content/drive/MyDrive/datasets/Fake_News_Classification_processed.csv"
}

def new_train_loop(dataset_name, output_path, n_splits=5):
    dataset_path = DATASETS[dataset_name]

    # Data Loading
    print(f"\nInitialising experiment on: {dataset_name}")
    df = pd.read_csv(dataset_path).dropna()
    print(f"✓ Dataset loaded: {len(df)} rows")
    print(f"Label distribution:\n{df['label'].value_counts()}")

    # 85-15 split: hold out 15% as the test set
    print("\n" + "=" * 60)
    print("TRAIN / TEST SPLIT (85-15)")
    print("=" * 60)

    train_texts, test_texts, train_labels, test_labels = train_test_split(
        df['combined_text'].tolist(),
        df['label'].tolist(),
        test_size=0.15,
        random_state=42,
        stratify=df['label']
    )
    print(f"✓ Training pool samples: {len(train_texts)} (85%)")
    print(f"✓ Test set samples:      {len(test_texts)} (15%)")

    # Tokenization
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

    print(f"\nTokenizing training pool for {n_splits}-fold Cross-Validation...")
    train_encodings = tokenizer(
        train_texts,
        truncation=True,
        padding=True,
        max_length=128
    )
    print("✓ Training pool tokenized")

    print("Tokenizing held-out test set...")
    test_encodings = tokenizer(
        test_texts,
        truncation=True,
        padding=True,
        max_length=128
    )
    print("✓ Test set tokenized")

    train_dataset = FakeNewsDataset(train_encodings, train_labels)
    test_dataset  = FakeNewsDataset(test_encodings, test_labels)

    # 5-Fold Cross Validation on the 85% training pool
    model, device, use_tpu = load_model()
    best_model = cross_validate_finetuning(
        train_dataset,
        test_dataset,
        compute_metrics,
        use_tpu,
        device,
        n_splits=n_splits
    )

    # Save best model
    save_model(best_model, tokenizer, output_path, use_tpu, device)

    # Generalization Testing (Against all other datasets)
    cross_dataset_evaluation(best_model, tokenizer, dataset_name, DATASETS, compute_metrics)

# WELFake Dataset

In [ ]:
new_train_loop("WELFake", "/content/drive/MyDrive/bert_models/WELFake")


Initialising experiment on: WELFake
✓ Dataset loaded: 63670 rows
Label distribution:
label
0    34790
1    28880
Name: count, dtype: int64

TRAIN / TEST SPLIT (85-15)
✓ Training pool samples: 54119 (85%)
✓ Test set samples:      9551 (15%)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.9992, F1: 0.9991

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.0188, F1: 0.0357


: 

In [ ]:
# train_loop("/content/drive/MyDrive/datasets/WELFake_processed.csv", "/content/drive/MyDrive/bert_models/WELFake")

: 

# FakeNewsNet

In [ ]:
new_train_loop("FakeNewsNet", "/content/drive/MyDrive/bert_models/FakeNewsNet")


Initialising experiment on: FakeNewsNet
✓ Dataset loaded: 21844 rows
Label distribution:
label
1    16522
0     5322
Name: count, dtype: int64

TRAIN / TEST SPLIT (85-15)
✓ Training pool samples: 18567 (85%)
✓ Test set samples:      3277 (15%)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.6210, F1: 0.5123

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.3813, F1: 0.2848


: 

In [ ]:
# train_loop("/content/drive/MyDrive/datasets/FakeNewsNet_processed.csv", "/content/drive/MyDrive/bert_models/FakeNewsNet")

: 

# Fake News Detection Dataset

In [ ]:
new_train_loop("Fake_News_Detection", "/content/drive/MyDrive/bert_models/Fake_News_Detection")


Initialising experiment on: Fake_News_Detection
✓ Dataset loaded: 38650 rows
Label distribution:
label
1    21194
0    17456
Name: count, dtype: int64

TRAIN / TEST SPLIT (85-15)
✓ Training pool samples: 32852 (85%)
✓ Test set samples:      5798 (15%)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.0002, F1: 0.0003

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.9822, F1: 0.9833


: 

In [ ]:
# train_loop("/content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv", "/content/drive/MyDrive/bert_models/Fake_News_Detection")

: 

# ISOT Dataset

In [ ]:
new_train_loop("ISOT", "/content/drive/MyDrive/bert_models/ISOT")


Initialising experiment on: ISOT
✓ Dataset loaded: 39098 rows
Label distribution:
label
0    21196
1    17902
Name: count, dtype: int64

TRAIN / TEST SPLIT (85-15)
✓ Training pool samples: 33233 (85%)
✓ Test set samples:      5865 (15%)

Tokenizing training pool for 5-fold Cross-Validation...
✓ Training pool tokenized
Tokenizing held-out test set...
✓ Test set tokenized
✓ Using TPU: xla:0

MODEL LOADING


/tmp/ipykernel_730/197324397.py:7: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_730/197324397.py:7: De

✓ BERT model loaded (TPU will handle device placement)

STARTING 5-FOLD CROSS VALIDATION (Pre-tokenized Data)

--- FOLD 1 ---
✓ Using TPU: xla:0

MODEL LOADING


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Detection Accuracy: 0.5414, F1: 0.7025

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.0186, F1: 0.0366


: 

In [ ]:
#train_loop("/content/drive/MyDrive/datasets/ISOT_processed.csv", "/content/drive/MyDrive/bert_models/ISOT")

: 

# Fake News Classification Dataset

In [ ]:
# train_loop("/content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv", "/content/drive/MyDrive/bert_models/Fake_News_Detection")

: 

In [ ]:
new_train_loop("Fake_News_Classification", "/content/drive/MyDrive/bert_models/Fake_News_Classification")


Initialising experiment on: Fake_News_Classification
✓ Dataset loaded: 40580 rows
Label distribution:
label
1    21923
0    18657
Name: count, dtype: int64

TRAIN / TEST SPLIT (85-15)
✓ Training pool samples: 34493 (85%)
✓ Test set samples:      6087 (15%)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Detection Accuracy: 0.4854, F1: 0.1162

Testing on unseen dataset: ISOT (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.0005, F1: 0.0003


: 

In [ ]:
import os
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

# 1. Extended evaluation function to capture and print Precision and Recall
def cross_dataset_evaluation_extended(model, tokenizer, current_dataset_name, all_datasets_paths, compute_metrics_fn):
    print("\n" + "!" * 60)
    print(f"CROSS-DATASET GENERALIZATION: {current_dataset_name}")
    print("!" * 60)

    results = {}

    for name, path in all_datasets_paths.items():
        if name == current_dataset_name:
            continue

        if not os.path.exists(path):
            print(f"Warning: Dataset path {path} not found. Skipping {name}...")
            continue

        print(f"\nTesting on unseen dataset: {name} (Full Dataset)...")
        df = pd.read_csv(path).dropna()

        test_texts = df['combined_text'].tolist()
        test_labels = df['label'].tolist()

        encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)
        dataset = FakeNewsDataset(encodings, test_labels)

        # Trainer for evaluation with larger batch size for speed
        eval_trainer = Trainer(
            model=model,
            compute_metrics=compute_metrics_fn,
            args=TrainingArguments(
                output_dir="./temp_eval",
                remove_unused_columns=False,
                label_names=["labels"],
                per_device_eval_batch_size=128,
                report_to="none"
            )
        )

        metrics = eval_trainer.evaluate(eval_dataset=dataset)
        results[name] = metrics
        
        acc = metrics.get('eval_accuracy', 0)
        f1 = metrics.get('eval_f1', 0)
        prec = metrics.get('eval_precision', 0)
        rec = metrics.get('eval_recall', 0)
        
        print(f"  -> {name} Accuracy: {acc:.4f}, F1: {f1:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}")

    return results

# 2. Paths to your saved baseline model directories
BASELINE_BERT_MODELS = {
    # "WELFake": "/content/drive/MyDrive/bert_models/WELFake",
    # "FakeNewsNet": "/content/drive/MyDrive/bert_models/FakeNewsNet",
    "Fake_News_Detection": "/content/drive/MyDrive/bert_models/Fake_News_Detection",
    "ISOT": "/content/drive/MyDrive/bert_models/ISOT",
    "Fake_News_Classification": "/content/drive/MyDrive/bert_models/Fake_News_Classification"
}

# 3. Main evaluation loop
all_baseline_results = {}

for model_name, model_dir in BASELINE_BERT_MODELS.items():
    if not os.path.exists(model_dir):
        print(f"\nSkipping model '{model_name}': directory '{model_dir}' not found.")
        continue
        
    print(f"\n==================================================")
    print(f"LOADING BASELINE BERT MODEL TRAINED ON: {model_name}")
    print(f"==================================================")
    
    # Load model and tokenizer from local Google Drive weights
    model = BertForSequenceClassification.from_pretrained(model_dir)
    tokenizer = BertTokenizer.from_pretrained(model_dir)
    
    # If device was setup in prior cells, move model to GPU/TPU
    if 'device' in globals() and device is not None:
        model.to(device)
    
    # Run evaluation
    results = cross_dataset_evaluation_extended(
        model=model,
        tokenizer=tokenizer,
        current_dataset_name=model_name,
        all_datasets_paths=DATASETS,  # Uses the DATASETS map defined in Section 8 of your notebook
        compute_metrics_fn=compute_metrics  # Uses your notebook's compute_metrics function
    )
    all_baseline_results[model_name] = results

# 4. Print beautiful consolidated summary table
print("\n" + "=" * 90)
print("BASELINE BERT CROSS-DATASET GENERALIZATION EVALUATION SUMMARY")
print("=" * 90)
print(f"{'Trained On':<22} | {'Tested On':<22} | {'Accuracy':<10} | {'F1':<10} | {'Precision':<10} | {'Recall':<10}")
print("-" * 105)
for trained_on, evaluations in all_baseline_results.items():
    for tested_on, metrics in evaluations.items():
        acc = metrics.get('eval_accuracy', 0)
        f1 = metrics.get('eval_f1', 0)
        prec = metrics.get('eval_precision', 0)
        rec = metrics.get('eval_recall', 0)
        print(f"{trained_on:<22} | {tested_on:<22} | {acc:.4f}     | {f1:.4f} | {prec:.4f}    | {rec:.4f}")



LOADING BASELINE BERT MODEL TRAINED ON: Fake_News_Detection


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
CROSS-DATASET GENERALIZATION: Fake_News_Detection
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

Testing on unseen dataset: WELFake (Full Dataset)...


  -> WELFake Accuracy: 0.1350, F1: 0.0329, Precision: 0.0333, Recall: 0.0324

Testing on unseen dataset: FakeNewsNet (Full Dataset)...


  -> FakeNewsNet Accuracy: 0.3666, F1: 0.3308, Precision: 0.8237, Recall: 0.2069

Testing on unseen dataset: ISOT (Full Dataset)...


...

  -> WELFake Accuracy: 0.1415, F1: 0.0217, Precision: 0.0224, Recall: 0.0209

Testing on unseen dataset: FakeNewsNet (Full Dataset)...


  -> FakeNewsNet Accuracy: 0.2597, F1: 0.0548, Precision: 0.7976, Recall: 0.0284

Testing on unseen dataset: Fake_News_Detection (Full Dataset)...


  -> Fake_News_Detection Accuracy: 0.4853, F1: 0.1159, Precision: 0.9969, Recall: 0.0615

Testing on unseen dataset: ISOT (Full Dataset)...


  -> ISOT Accuracy: 0.0005, F1: 0.0003, Precision: 0.0002, Recall: 0.0003

BASELINE BERT CROSS-DATASET GENERALIZATION EVALUATION SUMMARY
Trained On             | Tested On              | Accuracy   | F1         | Precision  | Recall    
---------------------------------------------------------------------------------------------------------
Fake_News_Detection    | WELFake                | 0.1350     | 0.0329 | 0.0333    | 0.0324
Fake_News_Detection    | FakeNewsNet            | 0.3666     | 0.3308 | 0.8237    | 0.2069
Fake_News_Detection    | ISOT                   | 0.0002     | 0.0003 | 0.0003    | 0.0003
Fake_News_Detection    | Fake_News_Classification | 0.9822     | 0.9833 | 0.9982    | 0.9688
ISOT                   | WELFake                | 0.8434     | 0.8517 | 0.7467    | 0.9909
ISOT                   | FakeNewsNet            | 0.7558     | 0.8609 | 0.7565    | 0.9987
ISOT                   | Fake_News_Detection    | 0.5414     | 0.7025 | 0.5452    | 0.9873
ISOT              

: 

In [11]:
import os
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

# 1. Extended evaluation function to capture and print Precision and Recall
def cross_dataset_evaluation_extended(model, tokenizer, current_dataset_name, all_datasets_paths, compute_metrics_fn):
    print("\n" + "!" * 60)
    print(f"CROSS-DATASET GENERALIZATION: {current_dataset_name}")
    print("!" * 60)

    results = {}

    for name, path in all_datasets_paths.items():
        if name == current_dataset_name:
            continue

        if not os.path.exists(path):
            print(f"Warning: Dataset path {path} not found. Skipping {name}...")
            continue

        print(f"\nTesting on unseen dataset: {name} (Full Dataset)...")
        df = pd.read_csv(path).dropna()

        test_texts = df['combined_text'].tolist()
        test_labels = df['label'].tolist()

        encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)
        dataset = FakeNewsDataset(encodings, test_labels)

        # Trainer for evaluation with larger batch size for speed
        eval_trainer = Trainer(
            model=model,
            compute_metrics=compute_metrics_fn,
            args=TrainingArguments(
                output_dir="./temp_eval",
                remove_unused_columns=False,
                label_names=["labels"],
                per_device_eval_batch_size=128,
                report_to="none"
            )
        )

        metrics = eval_trainer.evaluate(eval_dataset=dataset)
        results[name] = metrics
        
        acc = metrics.get('eval_accuracy', 0)
        f1 = metrics.get('eval_f1', 0)
        prec = metrics.get('eval_precision', 0)
        rec = metrics.get('eval_recall', 0)
        
        print(f"  -> {name} Accuracy: {acc:.4f}, F1: {f1:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}")

    return results

# 2. Paths to your saved baseline model directories
BASELINE_BERT_MODELS = {
    # "WELFake": "/content/drive/MyDrive/bert_models/WELFake",
    "FakeNewsNet": "/content/drive/MyDrive/bert_models/FakeNewsNet",
    "Fake_News_Detection": "/content/drive/MyDrive/bert_models/Fake_News_Detection",
    "ISOT": "/content/drive/MyDrive/bert_models/ISOT",
    "Fake_News_Classification": "/content/drive/MyDrive/bert_models/Fake_News_Classification"
}

# 3. Main evaluation loop
all_baseline_results = {}

for model_name, model_dir in BASELINE_BERT_MODELS.items():
    if not os.path.exists(model_dir):
        print(f"\nSkipping model '{model_name}': directory '{model_dir}' not found.")
        continue
        
    print(f"\n==================================================")
    print(f"LOADING BASELINE BERT MODEL TRAINED ON: {model_name}")
    print(f"==================================================")
    
    # Load model and tokenizer from local Google Drive weights
    model = BertForSequenceClassification.from_pretrained(model_dir)
    tokenizer = BertTokenizer.from_pretrained(model_dir)
    
    # If device was setup in prior cells, move model to GPU/TPU
    if 'device' in globals() and device is not None:
        model.to(device)
    
    # Run evaluation
    results = cross_dataset_evaluation_extended(
        model=model,
        tokenizer=tokenizer,
        current_dataset_name=model_name,
        all_datasets_paths=DATASETS,  # Uses the DATASETS map defined in Section 8 of your notebook
        compute_metrics_fn=compute_metrics  # Uses your notebook's compute_metrics function
    )
    all_baseline_results[model_name] = results

# 4. Print beautiful consolidated summary table
print("\n" + "=" * 90)
print("BASELINE BERT CROSS-DATASET GENERALIZATION EVALUATION SUMMARY")
print("=" * 90)
print(f"{'Trained On':<22} | {'Tested On':<22} | {'Accuracy':<10} | {'F1':<10} | {'Precision':<10} | {'Recall':<10}")
print("-" * 105)
for trained_on, evaluations in all_baseline_results.items():
    for tested_on, metrics in evaluations.items():
        acc = metrics.get('eval_accuracy', 0)
        f1 = metrics.get('eval_f1', 0)
        prec = metrics.get('eval_precision', 0)
        rec = metrics.get('eval_recall', 0)
        print(f"{trained_on:<22} | {tested_on:<22} | {acc:.4f}     | {f1:.4f} | {prec:.4f}    | {rec:.4f}")



LOADING BASELINE BERT MODEL TRAINED ON: FakeNewsNet


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
CROSS-DATASET GENERALIZATION: FakeNewsNet
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

Testing on unseen dataset: WELFake (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> WELFake Accuracy: 0.5709, F1: 0.5071, Precision: 0.5293, Recall: 0.4866

Testing on unseen dataset: Fake_News_Detection (Full Dataset)...


...

  -> FakeNewsNet Accuracy: 0.2597, F1: 0.0550, Precision: 0.7970, Recall: 0.0285

Testing on unseen dataset: Fake_News_Detection (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Detection Accuracy: 0.4854, F1: 0.1162, Precision: 0.9970, Recall: 0.0617

Testing on unseen dataset: ISOT (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.0005, F1: 0.0003, Precision: 0.0002, Recall: 0.0003

BASELINE BERT CROSS-DATASET GENERALIZATION EVALUATION SUMMARY
Trained On             | Tested On              | Accuracy   | F1         | Precision  | Recall    
---------------------------------------------------------------------------------------------------------
FakeNewsNet            | WELFake                | 0.5709     | 0.5071 | 0.5293    | 0.4866
FakeNewsNet            | Fake_News_Detection    | 0.4243     | 0.3687 | 0.4624    | 0.3065
FakeNewsNet            | ISOT                   | 0.6210     | 0.5123 | 0.6235    | 0.4347
FakeNewsNet            | Fake_News_Classification | 0.3813     | 0.2848 | 0.3793    | 0.2280
Fake_News_Detection    | WELFake                | 0.1351     | 0.0329 | 0.0334    | 0.0325
Fake_News_Detection    | FakeNewsNet            | 0.3670     | 0.3315 | 0.8237    | 0.2075
Fake_News_Detection    | ISOT                   | 0.0002     | 0.0003 | 0.0003    | 0.0003
Fake_News_Detectio

In [13]:
all_baseline_results = {}

WELFAKE = {
    "WELFake": "/content/drive/MyDrive/bert_models/WELFake",
}

for model_name, model_dir in WELFAKE.items():
    if not os.path.exists(model_dir):
        print(f"\nSkipping model '{model_name}': directory '{model_dir}' not found.")
        continue
        
    print(f"\n==================================================")
    print(f"LOADING BASELINE BERT MODEL TRAINED ON: {model_name}")
    print(f"==================================================")
    
    # Load model and tokenizer from local Google Drive weights
    model = BertForSequenceClassification.from_pretrained(model_dir)
    tokenizer = BertTokenizer.from_pretrained(model_dir)
    
    # If device was setup in prior cells, move model to GPU/TPU
    if 'device' in globals() and device is not None:
        model.to(device)
    
    # Run evaluation
    results = cross_dataset_evaluation_extended(
        model=model,
        tokenizer=tokenizer,
        current_dataset_name=model_name,
        all_datasets_paths=DATASETS,  # Uses the DATASETS map defined in Section 8 of your notebook
        compute_metrics_fn=compute_metrics  # Uses your notebook's compute_metrics function
    )
    all_baseline_results[model_name] = results

# 4. Print beautiful consolidated summary table
print("\n" + "=" * 90)
print("BASELINE BERT CROSS-DATASET GENERALIZATION EVALUATION SUMMARY")
print("=" * 90)
print(f"{'Trained On':<22} | {'Tested On':<22} | {'Accuracy':<10} | {'F1':<10} | {'Precision':<10} | {'Recall':<10}")
print("-" * 105)
for trained_on, evaluations in all_baseline_results.items():
    for tested_on, metrics in evaluations.items():
        acc = metrics.get('eval_accuracy', 0)
        f1 = metrics.get('eval_f1', 0)
        prec = metrics.get('eval_precision', 0)
        rec = metrics.get('eval_recall', 0)
        print(f"{trained_on:<22} | {tested_on:<22} | {acc:.4f}     | {f1:.4f} | {prec:.4f}    | {rec:.4f}")



LOADING BASELINE BERT MODEL TRAINED ON: WELFake


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
CROSS-DATASET GENERALIZATION: WELFake
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

Testing on unseen dataset: FakeNewsNet (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> FakeNewsNet Accuracy: 0.6984, F1: 0.8165, Precision: 0.7562, Recall: 0.8872

Testing on unseen dataset: Fake_News_Detection (Full Dataset)...


...

  -> Fake_News_Detection Accuracy: 0.2231, F1: 0.3640, Precision: 0.3302, Recall: 0.4054

Testing on unseen dataset: ISOT (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.9992, F1: 0.9991, Precision: 0.9998, Recall: 0.9983

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.0188, F1: 0.0357, Precision: 0.0380, Recall: 0.0336

BASELINE BERT CROSS-DATASET GENERALIZATION EVALUATION SUMMARY
Trained On             | Tested On              | Accuracy   | F1         | Precision  | Recall    
---------------------------------------------------------------------------------------------------------
WELFake                | FakeNewsNet            | 0.6984     | 0.8165 | 0.7562    | 0.8872
WELFake                | Fake_News_Detection    | 0.2231     | 0.3640 | 0.3302    | 0.4054
WELFake                | ISOT                   | 0.9992     | 0.9991 | 0.9998    | 0.9983
WELFake                | Fake_News_Classification | 0.0188     | 0.0357 | 0.0380    | 0.0336
